# Сравнение `agr_id` из Excel и витрины

Тетрадка сравнивает множества `agr_id`:
- из Excel-отчета,
- из витрины `sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart` за выбранный месяц.

Результат:
- `intersect_cnt`
- `only_excel_cnt`
- `only_datamart_cnt`

In [ ]:
import pandas as pd

excel_path = "/home/jovyan/documents/Equaring/Проверки/02_февраль.xlsx"  # путь к Excel
agr_col = "agr_id"          # имя колонки agr_id в Excel
month_start = "2026-02-01"  # YYYY-MM-01

In [ ]:
# 1) agr_id из Excel
df = pd.read_excel(excel_path)

excel_agr = (
    df[agr_col]
    .astype(str)
    .str.strip()
    .replace({"": None, "nan": None, "None": None})
    .dropna()
    .str.replace(r"\\.0$", "", regex=True)
    .drop_duplicates()
)

print("excel_rows =", len(df))
print("excel_unique_agr =", len(excel_agr))
excel_agr.head()

In [ ]:
# 2) agr_id из витрины за месяц
with imp:
    dm = imp.fetch(f"""
        select distinct cast(agr_id as string) as agr_id
        from sandbox_ai.stg__chesnov_aef__sa_acquiring_datamart
        where trx_month = cast('{month_start}' as date)
          and agr_id is not null
    """)

dm_agr = (
    dm["agr_id"]
    .astype(str)
    .str.strip()
    .replace({"": None, "nan": None, "None": None})
    .dropna()
    .str.replace(r"\\.0$", "", regex=True)
    .drop_duplicates()
)

print("datamart_unique_agr =", len(dm_agr))
dm_agr.head()

In [ ]:
# 3) Сравнение множеств
set_excel = set(excel_agr.tolist())
set_dm = set(dm_agr.tolist())

intersect = set_excel & set_dm
only_excel = set_excel - set_dm
only_dm = set_dm - set_excel

summary = pd.DataFrame([
    {"metric": "excel_unique_agr", "value": len(set_excel)},
    {"metric": "datamart_unique_agr", "value": len(set_dm)},
    {"metric": "intersect_cnt", "value": len(intersect)},
    {"metric": "only_excel_cnt", "value": len(only_excel)},
    {"metric": "only_datamart_cnt", "value": len(only_dm)},
])
summary

In [ ]:
# 4) Примеры расхождений
only_excel_df = pd.DataFrame({"agr_id": sorted(list(only_excel))})
only_datamart_df = pd.DataFrame({"agr_id": sorted(list(only_dm))})

print("Examples: only in Excel")
display(only_excel_df.head(50))

print("Examples: only in Datamart")
display(only_datamart_df.head(50))